[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nube1/h-potential-companions/blob/main/methods-series/notebooks/04_dynamics.ipynb)

# Paper 4: verifying the solvers, and finding out what the data can identify

A continuous-time model of health state is only as trustworthy as the code that solves it.
Four checks, each of which can fail: the order of the deterministic solver, the strong
order of the stochastic one under two noise structures, conservation of probability mass in
the density solver, and whether an observation design can identify the diffusion at all.

## Status, before anything else

- **Every datum here is synthetic.** Nothing in this notebook was measured on a person.
- **Papers 1 to 3 are protocols.** They say what will be done, not what was found. No
  participant has been recruited and no Delphi panel has met.
- **Papers 4 to 6 are method illustrations.** Their inputs show a workflow. They are not
  estimates from any health system and not claims about any product.
- **No figure below is evidence about patients.** Each number is a property of a stated
  model or a stated decision rule, and nothing more.

## Setup

The cell below installs the package straight from GitHub when this runs in Colab, and installs the local checkout when it does not. It prints the resolved version so you know what you are running.

**Runtime.** Reduced run: about 50 seconds. Full run: about ninety seconds.

In [ ]:
# ---------------------------------------------------------------- setup
# Repository holding this package. The repository is private: in Colab, sign
# in to GitHub or clone it yourself before running.
GITHUB_OWNER = "Nube1"
REPO = "h-potential-companions"
SUBDIR = "methods-series"

import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q "hpotential[full] @ git+https://github.com/{GITHUB_OWNER}/{REPO}.git#subdirectory={SUBDIR}"
else:
    # Running from a clone: install the checkout this notebook sits in.
    import importlib.util
    import pathlib
    import subprocess

    if importlib.util.find_spec("hpotential") is None:
        for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
            if (parent / "src" / "hpotential").exists():
                subprocess.run(
                    [sys.executable, "-m", "pip", "install", "-q", "-e", str(parent)], check=True
                )
                break
        else:
            raise RuntimeError(
                "hpotential is not importable and no local checkout was found. "
                "Run this in Colab with access to the repository, or clone it."
            )

import hpotential

print(f"hpotential {hpotential.__version__}")
print(f"running in Colab: {IN_COLAB}")

In [ ]:
# Importing hpotential selects the Agg backend, because the library writes figures to
# files. Put the plots back into the page, then borrow the series figure style.
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hpotential.utils.plotting import GRID, INK, apply_style, series_color

apply_style()
plt.rcParams["figure.dpi"] = 110  # 200 is for print; this is for a screen

try:
    import ipywidgets as widgets
    from IPython.display import display

    HAVE_WIDGETS = True
except Exception:  # Colab sometimes ships without a working widget stack
    HAVE_WIDGETS = False


def explore(fn, controls, defaults):
    """Draw the static case first, then attach sliders if ipywidgets works.

    The static call means the notebook still teaches something when it is read on
    GitHub, where no widget ever runs.
    """
    fn(**defaults)
    if HAVE_WIDGETS:
        display(widgets.interactive(fn, **controls))
    else:
        print("ipywidgets is unavailable here, so the static case above is all there is.")


print("plots inline, style applied, widgets:", HAVE_WIDGETS)

In [ ]:
# ---------------------------------------------------------------- run size
FULL_RUN = False   # True reproduces the published values. Full run here takes about ninety seconds.


def banner():
    """Print at the top of every results cell. It has to be impossible to miss."""
    if FULL_RUN:
        print("=" * 74)
        print("FULL RUN: published replicate counts. These are the published values.")
        print("=" * 74)
    else:
        print("=" * 74)
        print("REDUCED RUN: cut replicate counts for speed.")
        print("The numbers below are NOT the published values. Set FULL_RUN = True.")
        print("=" * 74)


banner()

## The process

A mean-reverting state with either additive or multiplicative noise,

$$dX_t = (a - bX_t)\,dt + \sigma\,dW_t
\qquad\text{or}\qquad
dX_t = (a - bX_t)\,dt + \sigma X_t\,dW_t ,$$

with $a = 1.2$, $b = 0.6$, $\sigma = 0.45$, $X_0 = 3$ and $T = 6$. The deterministic part
has the closed-form solution

$$x(t) = \frac{a}{b} + \left(x_0 - \frac{a}{b}\right)e^{-bt},$$

so errors can be measured against the truth rather than against a finer run of the same
scheme.

In [ ]:
banner()

from hpotential.core.dynamical import (
    FokkerPlanck, LinearSDE, euler_maruyama, profile_sigma, rk4, simulate_state,
    stable_time_step, strong_error_study, weak_error_study,
)
from hpotential.utils.numerical import convergence_order, pairwise_orders

A, B, SIGMA, X0, T = 1.2, 0.6, 0.45, 3.0, 6.0
sde = LinearSDE(A, B, SIGMA)
print(f"mean-reversion level a / b = {sde.mean_reversion_level:.4f}")
print(f"exact x(T)                 = {sde.exact_solution(T, X0):.10f}")

## Check 1: RK4 is fourth order

Halve the step and a fourth-order method should cut the error by a factor of 16. The
observed order is the slope of $\log(\text{error})$ against $\log(h)$.

In [ ]:
banner()

steps = np.array([0.4, 0.2, 0.1, 0.05, 0.025, 0.0125])
exact = sde.exact_solution(T, X0)
errors = np.array([abs(rk4(lambda t, x: sde.drift(x), X0, T, h) - exact) for h in steps])

print(f"{'h':>8} {'error':>14} {'ratio':>10} {'local order':>12}")
prev = None
for h, e in zip(steps, errors):
    ratio = "" if prev is None else f"{prev / e:10.2f}"
    order = "" if prev is None else f"{np.log2(prev / e):12.3f}"
    print(f"{h:8.4f} {e:14.4e} {ratio:>10} {order:>12}")
    prev = e
print()
print(f"fitted order over the whole sweep: {convergence_order(steps, errors):.3f}")

## Check 2: strong order, and why the noise structure decides it

Strong error is $\sqrt{E|X^h_T - X_T|^2}$, pathwise. The reference and every coarse
solution are driven by the same Brownian increments, so what is measured is discretisation
error and not sampling noise.

Euler-Maruyama has strong order 1 when the diffusion does not depend on the state, because
the Milstein correction term vanishes. It drops to order $1/2$ when the diffusion is
$\sigma X_t$. Both cases are run because the difference is the whole point.

In [ ]:
banner()

import time

n_paths = 4000 if FULL_RUN else 1000
counts = [75, 150, 300, 600, 1200]

start = time.time()
studies = {}
for index, multiplicative in enumerate((False, True)):
    process = LinearSDE(A, B, SIGMA, multiplicative=multiplicative)
    label = "multiplicative" if multiplicative else "additive"
    strong = strong_error_study(process, X0, T, counts, n_paths, np.random.default_rng([2026, index]), 9600)
    weak = weak_error_study(process, X0, T, counts)
    studies[label] = {**strong, **weak}
    print(f"{label:<15} strong order {strong['strong_order']:.3f}   weak order {weak['weak_order']:.3f}")
print(f"{n_paths} paths per step size, {time.time() - start:.1f} s")

In [ ]:
banner()

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))

ax = axes[0]
for i, (label, s) in enumerate(studies.items()):
    ax.loglog(s["steps"], s["strong_errors"], "o-", ms=5, lw=1.5, color=series_color(i),
              label=f"{label}, fitted {s['strong_order']:.2f}")
h = np.array(studies["additive"]["steps"])
ax.loglog(h, errors[0] * 0 + h * (studies["additive"]["strong_errors"][0] / h[0]), ls="--",
          lw=0.9, color=INK, label="slope 1")
ax.loglog(h, np.sqrt(h) * (studies["multiplicative"]["strong_errors"][0] / np.sqrt(h[0])), ls=":",
          lw=0.9, color=INK, label="slope 1/2")
ax.set_xlabel("step $h$"); ax.set_ylabel(r"strong error $\sqrt{E|X^h_T - X_T|^2}$")
ax.set_title("Strong convergence", fontsize=10); ax.legend(fontsize=8)

ax = axes[1]
ax.loglog(steps, errors, "o-", ms=5, lw=1.5, color=series_color(2),
          label=f"RK4, fitted {convergence_order(steps, errors):.2f}")
ax.loglog(steps, errors[0] * (steps / steps[0]) ** 4, ls="--", lw=0.9, color=INK, label="slope 4")
ax.set_xlabel("step $h$"); ax.set_ylabel("error at $T$")
ax.set_title("RK4 on the deterministic part", fontsize=10); ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

print("Monte Carlo standard errors on the terminal mean, additive case:")
print("  " + "  ".join(f"{v:.2e}" for v in studies["additive"]["mc_se"]))
print("These are larger than the weak errors, which is why the weak order is computed")
print("in closed form from the deterministic recursion rather than estimated from paths.")

## Check 3: mass conservation without renormalisation

The density $p(x,t)$ obeys the Fokker-Planck equation

$$\partial_t p = -\partial_x\!\left[(a - bx)p\right]
 + \tfrac{1}{2}\,\partial_x^2\!\left[\sigma^2 p\right],$$

which is a conservation law: written in flux form $\partial_t p = -\partial_x J$, with no
flux through the boundaries, total mass is constant for all time.

A finite-volume scheme that updates cells by the difference of face fluxes conserves mass
to machine precision by construction, because each interior flux is added to one cell and
subtracted from its neighbour. That is the check below. It matters that no renormalisation
is applied: dividing by the total at each step would produce a flat mass history from a
scheme that was losing mass, and hide the bug it was meant to reveal.

In [ ]:
banner()

grids = (100, 200, 400, 800) if FULL_RUN else (100, 200, 400)
t_end = 25.0

start = time.time()
fp = FokkerPlanck(sde, -2.0, 8.0)
rows, widths, l1 = [], [], []
for nx in grids:
    _, _, dx = fp.grid(nx)
    dt_cap = stable_time_step(sde, dx, -2.0, 8.0, safety=0.4)
    n_steps = int(np.ceil(t_end / dt_cap))
    sol = fp.solve(nx, n_steps, t_end, x0=X0)
    err = sol.l1_error(sde.stationary_pdf(sol.centres))
    widths.append(dx); l1.append(err)
    rows.append({"cells": nx, "dx": sol.dx, "dt": sol.dt, "steps": n_steps,
                 "mass drift": sol.mass_drift, "min density": sol.min_density,
                 "L1 vs stationary": err})
fp_table = pd.DataFrame(rows)
print(f"{time.time() - start:.1f} s")
print(f"L1 grid convergence order: {convergence_order(np.array(widths), np.array(l1)):.4f}")
fp_table

In [ ]:
banner()

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))

ax = axes[0]
sol = fp.solve(grids[-1], int(np.ceil(t_end / stable_time_step(sde, fp.grid(grids[-1])[2],
                                                               -2.0, 8.0, safety=0.4))),
               t_end, x0=X0)
ax.plot(sol.mass - 1.0, lw=1.2, color=series_color(0))
ax.set_xlabel("time step"); ax.set_ylabel("total mass minus one")
ax.set_title(f"Mass drift over {len(sol.mass)} steps, no renormalisation", fontsize=10)
ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))

ax = axes[1]
ax.loglog(widths, l1, "o-", ms=5, lw=1.5, color=series_color(1),
          label=f"fitted order {convergence_order(np.array(widths), np.array(l1)):.2f}")
ax.loglog(widths, np.array(l1)[0] * (np.array(widths) / widths[0]), ls="--", lw=0.9, color=INK,
          label="slope 1")
ax.set_xlabel(r"cell width $\Delta x$"); ax.set_ylabel("$L_1$ error against the stationary density")
ax.set_title("Grid convergence", fontsize=10); ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"largest mass drift over every grid: {fp_table['mass drift'].abs().max():.3e}")
print(f"smallest density anywhere:          {fp_table['min density'].min():.3e}   (never negative)")
print()
print("Machine precision on mass, first order in space on accuracy. The two are")
print("separate claims: a scheme can conserve mass exactly and still be inaccurate.")

## Check 4: what an observation design can identify

The last check is not about code. Suppose the state is observed with error at a handful of
visits. Can the diffusion $\sigma$ be recovered? Profile the exact Kalman likelihood over a
grid of $\sigma$, summing over subjects, and read the interval where the profile is within
1.92 log-likelihood units of its maximum.

Three designs: five visits with one indicator, five visits with three, and twenty-five
visits with three.

In [ ]:
banner()

n_subjects = 60 if FULL_RUN else 20
grid_n = 40 if FULL_RUN else 24
obs_sd = 0.35

start = time.time()
grid = np.linspace(0.12, 1.2, grid_n)
designs = (("sparse, 1 indicator", 5, 1), ("sparse, 3 indicators", 5, 3),
           ("dense, 3 indicators", 25, 3))
profiles, ident_rows = {}, []
for design_index, (label, n_visits, n_indicators) in enumerate(designs):
    times = np.linspace(0, T, n_visits)
    rng = np.random.default_rng([2026, 5, design_index])
    total = np.zeros_like(grid)
    for _ in range(n_subjects):
        y = simulate_state(sde, times, rng, obs_sd, n_indicators)
        total += profile_sigma(y, times, sde, grid)
    total -= total.max()
    profiles[label] = total
    inside = grid[total >= -1.92]
    ident_rows.append({"design": label, "visits": n_visits, "indicators": n_indicators,
                       "sigma hat": grid[int(np.argmax(total))],
                       "interval low": inside.min(), "interval high": inside.max(),
                       "width": inside.max() - inside.min()})
print(f"{n_subjects} subjects per design, {time.time() - start:.1f} s")
pd.DataFrame(ident_rows).round(4)

In [ ]:
banner()

fig, ax = plt.subplots(figsize=(6.6, 4.0))
for i, (label, profile) in enumerate(profiles.items()):
    ax.plot(grid, profile, lw=1.6, color=series_color(i), label=label)
ax.axhline(-1.92, ls="--", lw=0.9, color=INK)
ax.axvline(SIGMA, ls=":", lw=1.2, color=series_color(5))
ax.text(SIGMA + 0.01, ax.get_ylim()[0] * 0.55, rf"true $\sigma$ = {SIGMA}", fontsize=8,
        color=series_color(5))
ax.set_ylim(-30, 2)
ax.set_xlabel(r"$\sigma$"); ax.set_ylabel("profile log-likelihood, relative to its maximum")
ax.set_title("A flat profile is a design problem, not a solver problem", fontsize=10)
ax.legend(fontsize=8)
plt.show()

print("One indicator at five visits leaves a wide, flat interval: the data cannot")
print("separate process noise from measurement noise. Adding indicators at the same")
print("visits helps more than adding visits, because it pins the observation variance.")

---

## What this does not establish

**Verification is not validation.** Every check here says the code solves the equations it
claims to solve. None of them says the equations describe health.

**The observed orders are properties of this problem.** A different drift, a stiffer
regime, or a coarser grid outside the asymptotic range gives different numbers. Under a
reduced run the grids are deliberately coarser, so the fitted orders drift.

**Mass conservation is a statement about the scheme, not about accuracy.** The solver holds
mass to 1e-16 and still carries an $L_1$ error of a few per cent against the stationary
density at coarse grids. Those are two different claims and the notebook reports both.

**The identifiability result is about a design, not about a study.** Sixty synthetic
subjects with a known truth. It shows which designs could recover $\sigma$ if the model
were right, which is the necessary condition, not the sufficient one.

**A reduced run does not reproduce the published numbers.** In particular the fitted orders
become unreliable, because the coarse sweeps are not in the asymptotic regime.